### 単一元素結晶の分類問題


1. hcp (red)
2. bcc (blue)
3. fcc (green)

の３つの結晶構造を分類する問題である。


|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|<font color="red">H</font>| _ | _ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |<font color="red">He</font>|
|<font color="blue">Li</font>|<font color="red">Be</font>|_|_|_|_|_|_|_|_|_|_|B|C|N|O|F|<font color="green">Ne</font>|
|<font color="blue">Na</font>|<font color="red">Mg</font>|_|_|_|_|_|_|_|_|_|_|<font color="green">Al</font>|Si|P|S|Cl|<font color="green">Ar</font>|
|<font color="blue">K</font>|<font color="green">Ca</font>|<font color="red">Sc</font>|<font color="red">Ti</font>|<font color="blue">V</font>|<font color="blue">Cr</font>|Mn|<font color="blue">Fe</font>|<font color="red">Co</font>|<font color="green">Ni</font>|<font color="green">Cu</font>|<font color="red">Zn</font>|Ga|Ge|As|Se|Br|<font color="green">Kr</font>|
|<font color="blue">Rb</font>|<font color="green">Sr</font>|<font color="red">Y</font>|<font color="red">Zr</font>|<font color="blue">Nb</font>|<font color="blue">Mo</font>|<font color="red">Tc</font>|<font color="red">Ru</font>|<font color="green">Rh</font>|<font color="green">Pd</font>|<font color="green">Ag</font>|<font color="red">Cd</font>|In|Sn|Sb|Te|I|<font color="green">Xe</font>|
|<font color="blue">Cs</font>|<font color="blue">Ba</font>|_|<font color="red">Hf</font>|<font color="blue">Ta</font>|<font color="blue">W</font>|<font color="red">Re</font>|<font color="red">Os</font>|<font color="green">Ir</font>|<font color="green">Pt</font>|<font color="green">Au</font>|Hg|<font color="red">Tl</font>|<font color="green">Pb</font>|Bi|Po|At|Rn|
|Fr|Ra|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|
|_|_|La|<font color="green">Ce</font>|Pr|Nd|Pm|Sm|<font color="blue">Eu</font>|<font color="red">Gd</font>|<font color="red">Tb</font>|<font color="red">Dy</font>|<font color="red">Ho</font>|<font color="red">Er</font>|<font color="red">Tm</font>|<font color="green">Yb</font>|<font color="red">Lu</font>|_|
|_|_|<font color="green">Ac</font>|<font color="green">Th</font>|Pa|U|Np|Pu|Am|Cm|Bk|Cf|Es|Fm|Md|No|Lr|_|


**データ取得からデータ解析**

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.svm import LinearSVC, SVC
import warnings
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')


In [ ]:
# データ作成
g_dfraw = pd.read_csv("../data/mono_structure.csv")
g_descriptor_names = ['min_oxidation_state', 'max_oxidation_state', 'row',
                     'group', 's', 'p', 'd', 'f', 'atomic_radius_calculated', 'X', 'IP',
                     'EA']
g_target_name = 'crystal_structure'

def convert_crystaltype(dfraw, target_name,
                        target_str = {0: "misc", 1:"hcp", 2:"bcc", 3:"fcc"}):
    """       0: misc (black)
       1: hcp (red)
       2: bcc (blue)
       3: fcc (green)
       の変換を行う。

    Args:
        dfraw (pd.DataFrame): データ.
        target_name ([str]): 目的変数名
        target_str (dict, optional): 変換辞書. Defaults to {0: "misc", 1:"hcp", 2:"bcc", 3:"fcc"}.

    Returns:
        pd.DataFrame: 目的変数を変換されたデータ
    """
    targets = dfraw[target_name].values
    targetlist = []
    for target in targets:
        targetlist.append(target_str[target])
    targetlist
    dfraw[target_name] = targetlist
    return dfraw

g_dfraw =  convert_crystaltype(g_dfraw, g_target_name)
g_dfraw

簡単のために結晶構造が0: miscとなるデータを除く。


In [ ]:
# df = dfraw[dfraw["crystal_structure"] != 0].reset_index(drop=True)
g_df = g_dfraw[g_dfraw["crystal_structure"] != "misc"].reset_index(drop=True)

In [ ]:
def make_X_y(df, descriptor_names, target_name):
    """規格化を行いX,yを出力する。

    Args:
        df (pd.DataFrame): データ
        descriptor_names ([str]): 説明変数名
        target_name (str)): 目的変数名

    Returns:
        np.ndarray: 説明変数
        np.ndarray: 目的変数
    """
    Xraw = df.loc[:, descriptor_names].values
    y = df.loc[:, target_name].values

    # データプリプロセス
    scaler = StandardScaler()
    X = scaler.fit(Xraw)
    X = scaler.transform(Xraw)
    return X, y

g_X, g_y = make_X_y(g_df, g_descriptor_names, g_target_name)

## classificationの種類の設定

In [ ]:
REG_KIND = "LogisticRegression" # or "SVC" or "LinearSVC" or "LogisticRegression"

In [ ]:
def choose_cls(reg_kind, C):
    """分類インスタンスを作る。

    Args:
        reg_kind (str): 分類クラス名。
        C (float):  分類クラスハイパーパラメタ。

    Raises:
        ValueError: 規定外分類クラス名の場合。

    Returns:
        LinearSVC|SVC|LogisticRegression: 分類モデルインスタンス
    """
    if reg_kind == "LinearSVC":
        cls = LinearSVC(C=C) # 他にもパラメタはあるがdefaultを使う。
    elif reg_kind == "SVC":
        cls = SVC(C=C) # 他にもパラメタはあるがdefaultを使う。
    elif reg_kind == "LogisticRegression":
        cls = LogisticRegression(C=C)
    else:
        raise ValueError("unknown reg_kind, {}".format(reg_kind)) 
    return cls

In [ ]:
# データ解析
def CV_score(X, y, reg_kind, nfold=10):
    """X, y から分類スコアを得る。

    Args:
        X (np.ndarray): 説明変数。
        y (np.ndarray): 目的変数。
        reg_kind (str): classification model.
        nfold (int, optional): KFoldの分割数. Defaults to 10.

    Returns:
        [type]: [description]
    """
    Clist = np.logspace(-5, 3, 10)
    C_score_list = []
    for C in Clist:
        kf = KFold(nfold, shuffle=True)
        scorelist = []
        for train, test in kf.split(X):
            cls = choose_cls(reg_kind, C)

            Xtrain, ytrain = X[train], y[train]
            Xtest, ytest = X[test], y[test]
            cls.fit(Xtrain, ytrain)
            score = cls.score(Xtest, ytest)
            scorelist.append(score)
        scorelist = np.array(scorelist)
        C_score_list.append([C, scorelist.mean(), scorelist.std()])
    return pd.DataFrame(C_score_list, columns=["C","mean","std"])

g_df_score = CV_score(g_X, g_y, REG_KIND)

In [ ]:
g_df_score

In [ ]:
g_dfs = g_df_score.sort_values(by="mean", ascending=False).reset_index(drop=True)
g_Copt = g_dfs.loc[0,"C"]
g_Copt

### 観測データ全てを用いた評価指標値

In [ ]:
def make_model(X,y, reg_kind, Copt):
    """Coptを用いて、reg_kindから分類モデルを作り、全データで分類性能指標値を作る。

    Args:
        X (np.ndarray): 説明変数。
        y (np.ndarray): 目的変数。
        reg_kind (str): 分類モデル名。
        Copt (float): 分類モデルハイパーパラメタ。

    Returns:
        [type]: [description]
    """
    import os
    cls = choose_cls(reg_kind, Copt)

    cls.fit(X, y)
    yp = cls.predict(X)

    print("reg_kind", reg_kind)
    print("score", cls.score(X, y))
    print(classification_report(y, yp, digits=3))
    os.makedirs("image_executed", exist_ok=True)
    with open("image_executed/mono_structure_cls_report.txt", "w") as f:
        f.write(classification_report(y, yp, digits=3))

    index=[]
    columns= []
    for s in cls.classes_:
        index.append("actual({})".format(s))
        columns.append("predict({})".format(s))

    cmdf = pd.DataFrame(confusion_matrix(y, yp), index = index, columns=columns)
    display(cmdf)
    return y,yp, cls

g_y, g_yp, g_cls = make_model(g_X,g_y, REG_KIND, g_Copt)

### CV(test)の表示

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def CV_testscore(X, y, reg_kind, Copt, nfold=10):
    """CV testの評価を行う。

    Args:
        X (np.ndarray)): 説明変数
        y (np.ndarray): 目的変数
        reg_kind (str): classification type.
        Copt (float): C of logistic regression.
        nfold (int, optional): KFoldの分割数. Defaults to 10.

    Returns:
        pd.DataFrame: precision, recall, f1
        [np.ndarray]: a list of y_test
        [np.ndarray]: a list of predicted y_test

    """
    kf = KFold(nfold, shuffle=True)

    scorelist = []
    ylist = []
    yplist = []

    for train, test in kf.split(X):
        cls = choose_cls(reg_kind, Copt)

        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        cls.fit(Xtrain, ytrain)
        ytestp = cls.predict(Xtest)

        ylist.extend(ytest)
        yplist.extend(ytestp)
        
    # 各性能指標値のylistとyplistから求める。
    result = {}
    for average in  ["micro", "macro","weighted"]:
        result[average] = {"precision":precision_score(ylist,yplist, average=average).tolist(),
        "recall" : recall_score(ylist,yplist, average=average).tolist(),
        "f1": f1_score(ylist,yplist, average=average).tolist()}
    dfresultall= pd.DataFrame(result)
    
    return dfresultall, ylist, yplist

g_dfresultall, g_ylist,g_yplist = CV_testscore(g_X, g_y, REG_KIND, g_Copt, nfold=10)
display(g_dfresultall)


In [ ]:
def make_score(ylist, yplist, classes):
    """y, predicted y からclass毎のclassification scoreを作り表示する。

    Args:
        ylist (np.ndarry)): a list of y.
        yplist (np.ndarray): a list of predicted y.
        classes ([str]): cls classes_
    """
    average= None
    result = {"precision":precision_score(ylist,yplist, average=average).tolist(),
        "recall" : recall_score(ylist,yplist, average=average).tolist(),
        "f1": f1_score(ylist,yplist, average=average).tolist()}

    # indexはclsの中にある。
    dfresult = pd.DataFrame(result, index = classes)
    display(dfresult.T)
    
make_score(g_ylist, g_yplist, g_cls.classes_)

In [ ]:
# 次の表を書くための表示を行う。
np.set_printoptions(precision=3)
g_dfresultall["weighted"].values

### 予め計算した値

weighted averageの結果を示す。

|type | precision | recall | f1 |
|---|---|---|---|
| logistic regression |  0.658| 0.655| 0.656|
| SVC |  0.732| 0.707| 0.702|
| linearSVC | 0.614| 0.586| 0.588|

乱数依存があるので値は異なるかもしれません。


### 可視化

規格化された説明変数と目的変数を表示します。

実験値と予測値の比較をします。線が重なっていないところが予測に失敗している物質です。

In [ ]:
%matplotlib inline

def plot_y(y, y_predict, proba, symbols, labels):
    """plot y vs y_predict

    Args:
        y (np.array): target values.
        y_predict (np.array): predicted target values.
        proba (np.array): probability.
        symbols ([str]): 物質名。
        labels ([str]): 図示の表示ラベル。
    """
    plt.plot(y, "b-", label="y")
    plt.plot(y_predict, "r-", label="predict_y")
    plt.legend()
    plt.show()
   
    failedlist = []
    for i, (p1, p2, pro, s) in enumerate(zip(y, y_predict, proba, symbols)):
        if p1 != p2:
            failed = [s, p1, p2]
            failed.extend(pro)
            failedlist.append(failed)
            
    columns = ["element","actual","pred"]
    for i in labels:
        columns.append("P({})".format(i))
    print("failed at ")
    display(pd.DataFrame(failedlist, columns=columns))
            
if REG_KIND == "LogisticRegression":
    g_yproba = g_cls.predict_proba(g_X)    
    plot_y(g_y, g_yp, g_yproba, g_df["symbol"], g_cls.classes_)

## 平均値について

- micro average: TP FPを用いた平均値
- macro average: 単純な分類数での平均値
- weighted average: 分類数のサイズを加味した平均値

Ref. 
micro, macro, weighted averageについてはこちらに説明があります。

- https://androidkt.com/micro-macro-averages-for-imbalance-multiclass-classification/#:~:text=A%20macro%2Daverage%20will%20compute,to%20compute%20the%20average%20metric.

- https://datascience.stackexchange.com/questions/65839/macro-average-and-weighted-average-meaning-in-classification-report